# exp-020: 우위 재현 테스트 — 독립 평가 N확장(12명) + 통계 검정

- **목적:** exp-019(N=6, 통계검정 없음)의 약점을 보완. held-out **12명 × 후보 10 = 120쌍**으로 확장하고, per-user NDCG@10에 **Wilcoxon 부호순위 + 부트스트랩 95% CI + Cohen's d**를 적용해 "학습/황금 가중치가 임의 가중치보다 낫다"가 *통계적으로 재현*되는지 검정.
- **차별성 축:** ④ GT 부재 (독립 LLM 라벨 + 통계 검정) · ⑤ 자체 학습 노선 신뢰도
- **입력:** `raw/experiments/exp-020-independent-eval-stats/{eval_pairs.csv, llm_labels.csv}` (1~6번은 exp-019 라벨 재사용, 7~12번 신규 채점)
- **출력:** `raw/experiments/exp-020-independent-eval-stats/`
- **관련 위키:** exp-019-independent-llm-eval, exp-018-learnable-fusion-head
- **작성일:** 2026-05-31 · **시드:** 42 · **LLM 호출:** ❌ 0건 (LLM 세션 내 채점)

In [1]:
import pandas as pd, numpy as np, json
from pathlib import Path
from scipy import stats
ROOT=Path('.')
OUT=ROOT/'raw'/'experiments'/'exp-020-independent-eval-stats'
COMP=['role_match','hard_skill','industry_match','star_overlap','competency']
rng=np.random.default_rng(42)
df=pd.read_csv(OUT/'eval_pairs.csv').merge(pd.read_csv(OUT/'llm_labels.csv'),on='pair_id')
print(f'{df.userId.nunique()} users × {len(df)} pairs | label dist {df.llm_label.value_counts().sort_index().to_dict()}')

12 users × 120 pairs | label dist {0: 53, 1: 25, 2: 27, 3: 7, 4: 8}


In [2]:
def ndcg(scores,labels,k=10):
    o=np.argsort(-scores,kind='stable')[:k]; disc=1/np.log2(np.arange(2,k+2))
    dcg=(labels[o]*disc[:len(o)]).sum(); ideal=np.sort(labels)[::-1][:k]
    idcg=(ideal*disc[:len(ideal)]).sum(); return dcg/idcg if idcg>0 else np.nan
rows={'userId':[],'arb_E':[],'gold_E':[],'learn_E':[]}
for uid,g in df.groupby('userId'):
    L=g.llm_label.values.astype(float)
    if L.max()==0: continue
    rows['userId'].append(uid)
    for m in ['arb_E','gold_E','learn_E']: rows[m].append(ndcg(g[m].values.astype(float),L))
pu=pd.DataFrame(rows)
print('per-user NDCG@10 mean (n=%d):'%len(pu), {m:round(pu[m].mean(),4) for m in ['arb_E','gold_E','learn_E']})

per-user NDCG@10 mean (n=12): {'arb_E': np.float64(0.9204), 'gold_E': np.float64(0.9204), 'learn_E': np.float64(0.9184)}


In [3]:
def cohen_d(a,b):
    d=a-b; return float(d.mean()/d.std(ddof=1)) if d.std(ddof=1)>0 else 0.0
def boot_ci(a,b,nb=10000):
    d=(a-b); idx=np.arange(len(d)); ms=[d[rng.choice(idx,len(d),replace=True)].mean() for _ in range(nb)]
    return float(np.percentile(ms,2.5)),float(np.percentile(ms,97.5))
print('=== 우위 재현 통계 검정 (vs arbitrary, n=%d users) ==='%len(pu))
for m in ['gold_E','learn_E']:
    a,b=pu[m].values,pu['arb_E'].values; diff=a-b
    try: _,p=stats.wilcoxon(a,b)
    except Exception: p=np.nan
    lo,hi=boot_ci(a,b)
    print(f'  {m}-arbitrary: Δ={diff.mean():+.4f} | Wilcoxon p={p:.3f} | Cohen d={cohen_d(a,b):+.2f} | 95%CI[{lo:+.3f},{hi:+.3f}] | 0∈CI={lo<=0<=hi}')
print('\\n컴포넌트 vs LLM 독립 라벨 Spearman:')
for c in COMP: print(f'  {c:16s} {df[c].corr(df.llm_label,method="spearman"):+.3f}')

=== 우위 재현 통계 검정 (vs arbitrary, n=12 users) ===


.venv/lib/python3.12/site-packages/scipy/stats/_wilcoxon.py:181: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se


  gold_E-arbitrary: Δ=+0.0000 | Wilcoxon p=1.000 | Cohen d=+0.00 | 95%CI[+0.000,+0.000] | 0∈CI=True
  learn_E-arbitrary: Δ=-0.0019 | Wilcoxon p=0.945 | Cohen d=-0.04 | 95%CI[-0.030,+0.024] | 0∈CI=True
\n컴포넌트 vs LLM 독립 라벨 Spearman:
  role_match       +0.609
  hard_skill       +0.390
  industry_match   +0.331
  star_overlap     +0.247
  competency       +0.164


## 결과 (요약 — 상세는 exp-019-independent-llm-eval)

- **VERDICT: 우위 미재현.** 독립 라벨 120쌍에서 golden·learned 가중치가 arbitrary 대비 **통계적으로 유의한 우위 없음** (Wilcoxon p≈0.95, Cohen's d≈0, 95% CI가 0 포함).
- golden E = arbitrary E = 균등(.2×5)이라 Δ=정확히 0. learned는 Δ=−0.002로 사실상 동률(미세 열세).
- 컴포넌트 예측력 재확인: **role_match(+0.61) ≫ hard_skill(+0.39) > industry(+0.33) > star(+0.25) > competency(+0.16)**.
- **함의:** 5개 규칙 컴포넌트의 *선형 가중치 재조정*으로는 실제 적합도에서 임의값 이상을 못 낸다(N=12로 통계 확인). 성능 이득은 **임베딩(V2)** 에서 와야 함.

> 한계: N=12 user / 120쌍, 단일 채점자(LLM), holistic(E 관점) 라벨만. 관점별(A~D) 독립 라벨·인간 교차검증은 future work.